# EELSNMF Demo

In [1]:
import EELSNMF as enmf

cupy not available


In [2]:
# The main input for EELSNMF is hyperspy signal and the experimental conditions at which it was acquired:
import hyperspy.api as hs
s = hs.load(r"C:\Users\torruell\Documents\EPFL\eelsnmf tests\Test_SI.hspy")

# EELSNMF if a non-negative matrix factorization framework. The data need to be strictly positive:
s.data = np.clip(s.data,0,None)


# The experimental parameters are read if correctly defined in the image metadata.
deco = enmf.EELSNMF(s)
print(f"beam energy (V): {deco.E0}")
print(f"Convergence angle (rad): {deco.alpha}")
print(f"Collection angle (rad): {deco.beta}")

beam energy (V): 300000.0
Convergence angle (rad): 0.02022
Collection angle (rad): 0.036000000000000004


In [3]:
# For the physics guided approach we need to define the contributions to the measured EELS spectra, which are stored in the G matrix:

deco.build_G(
    low_loss = None, # currently the model only suports convolution with the average low-loss (not pixel-wise for a dual-EELS dataset)

    fine_structure_ranges = { #Here we define what are the edge that we expect in our EELS spectrum and their ELNES ranges (in eV)
        "Fe_L" : (704.,732.),
        "Cr_L" : (571.,597.),
        "O_K" : (526.,560.)
    }
)

deco.plot_energy_ranges()

L2 is used
L1 is used
L2 is used
L1 is used


At this stage we are ready for spectral decomposition.
There are different methods to choose depending on if smoothness and/or Sum-rule conservation want to be enforced.
The default method solves:

$$
\arg\min_{W,H}  \| X - G W H \|_F^2
$$

Using NMF multiplicative updates.

In [ ]:
deco.decomposition(
    n_components = 2,
    max_iters = 500,
    tol = 1e-9, # This tolerance defines the convergence condition
    decomposition_method = "default_decomposition"
    
)

In [7]:
#there are several built in plots to asses the results:
deco.plot_average_model() # plots the match between the average data spectrum and average model

In [8]:
deco.plot_factors() # plots the spectrum of each phase (the columns in GW)

In [10]:
deco.plot_loadings() # plots the abundance of each phase (the rows in H)

In [9]:
deco.plot_edges() # plots the ELNES associated to each element in each phase

In [ ]:
deco.plot_chemical_maps()

In [ ]:
deco.quantify_components()

# Kullback-Leibler divergence decomposition:

The EELS spectrum is counting events and therefore follows Poisson statistics. A metric better suited for this case is the Kullback-Leibler divergence. The library also implements NMF to solve:

$$
\arg\min_{W,H}  D_{KL}(X\|GWH)
$$

In [ ]:
deco.decomposition(
    n_components = 2,
    max_iters = 500,
    tol = 1e-9, 
    decomposition_method = "default_kl_decomposition",
    rescale_WH = True, # Fixes scale of H and W
    KL_rescaling_per_iter = True # KL divergence fits "variance", not absoulte scale. This corrects scale at each iteration (slow).
    
)

# Sum-rule decomposition:
Bethe sum-rules give some constrains on the intenstity distribution of a real cross-section. Within the model definied by the G matrix, this gives a constrain between the coefficients associated to the ELNES (f) and cross-section ($\sigma$) of a given edge (S) in spectral component k. This is enforced through the following objective function in the SumRule_decomposition method:

$$
\arg\min_{W,H}  \| X - G W H \|_F^2 + \frac{\lambda_{SR}}{2}\sum_{S,k} log\Big(\frac{\sum_{f_S}A_{f_S}W_{f_S}+\delta}{B_{S}W_{\sigma{S}}+\delta}\Big)^2
$$

In [ ]:
deco.decomposition(
    n_components = 2,
    max_iters = 500,
    tol = 1e-9, 
    decomposition_method = "SumRule_decomposition",
    lmbda = 1e3,
    norm = "none", #lmbda can be multiplied by the mean/elementwise G.T@X@H.T, the data weight in NMF updates, so that its easier to gauge a good value.
    SR_tolerance = 3, # sumrule penalty is not applied if its of by a factor less the SR_tolerance
    convergent_beam_correction = True, # correction used in the calculation of Bs, Afs.
    
    constrain = "elnes" , # penalty can be applied to cross-section coefficients, ELNES coefficients or both.
    # High lambda +"both" creates numerical instability.
    
    delta =1e-2, #coefficient for numerical stability.
)

# Edge Total Variation decomposition:

A realistic ELNES is broadenend by the spectral resolution of the system, resulting in a continuous curve. However, in the previous model there is nothing preventing consecutive $W_{f_S}$ of taking vastly different values. The EdgeTV decompositon also minimizes the second order total variation of the ELNES coefficient, resulting in smoother ELNES curves. in this context W,H are obtained through:

$$
\arg\min_{W,H}  \| X - G W H \|_F^2 + \lambda_{TV} \sum_{S,k}\sum_{i\in f_S} [(W_{i+1}-W_i)-(W_{i}-W_{i-1})]^2
$$

In [11]:
deco.decomposition(
    n_components = 2,
    max_iters = 500,
    tol = 1e-9, 
    decomposition_method = "EdgeTV_decomposition",
    lmbda = 1e5,
    norm = "none", #lmbda can be multiplied by the mean/elementwise G.T@X@H.T, the data weight in NMF updates, so that its easier to gauge a good value.
)

100%|████████████████████████████████████████| 500/500 [00:11<00:00, 43.47it/s, error=2.16e+6, relative change=3.92e-5]


# Combining Sum-rule and Total variation constraints:

This is achieved through the "combinedTVSR_decomposition" method:

In [4]:
deco.decomposition(
    n_components = 2,
    max_iters = 500,
    tol = 1e-9, 
    decomposition_method = "combinedTVSR_decomposition",
    TV_lmbda = 1e5,
    SR_lmbda = 1e3,
    norm = "none",
    SR_tolerance = 3, 
    convergent_beam_correction = True,
    constrain = "elnes" , 
    delta =1e-2

)

C:\Users\torruell\Documents\GitHub\EELSNMF\EELSNMF\utils.py:223: RuntimeWarning: invalid value encountered in arccos
  f = (1/np.pi)*(np.arccos(x)+((beta**2)/(alpha**2))*np.arccos(y)-(1/(2*alpha**2))*sqrt)
100%|███████████████████████████████████████████| 500/500 [00:10<00:00, 45.53it/s, error=9.57e+3, relative change=1.59]


# Additional functionalities:

* W_init,H_init to set initial W,H matrices close to desired solution
* Fixing coefficients: W_fixed_bool to fix values in W[W_fixed_bool] to W_fixed_values[W_fixed_bool].
* GPU acceleration through cupy (use_cupy=True)